# Lab 1.1 — Sandbox Setup & Data Profiling

**JuanMart** e-commerce sales data profiling notebook.

This notebook loads the raw `juanmart_raw_sales.csv` into pandas and runs a
full data profiling pass to surface anomalies before cleansing.

**Anomalies we expect to find:**
- Duplicate `transaction_id` rows
- Missing `cust_name` and `amount_paid` values
- Inconsistent region naming (`NCR`, `ncr`, `Metro Manila`, `Manila`, `CALABARZON`, `calabarzon`, `Region IV-A`)
- Mixed date formats (`YYYY-MM-DD` and `YYYY/MM/DD`)

In [7]:
import pandas as pd
import numpy as np
import json
import os

RAW_CSV = "juanmart_raw_sales.csv"

df = pd.read_csv(RAW_CSV)
df

,transaction_id,cust_name,region,order_date,amount_paid,status
0,1001,Juan Dela Cruz,NCR,2026-07-01,1500.50,Completed
1,1002,Maria Santos,Metro Manila,2026/07/02,2400.00,Completed
2,1003,NaN,ncr,2026-07-02,450.00,Cancelled
3,1004,Pedro Penduko,CALABARZON,2026/07/03,NaN,Completed
4,1005,Ana Roces,calabarzon,2026-07-04,3100.25,Completed
5,1001,Juan Dela Cruz,NCR,2026-07-01,1500.50,Completed
6,1006,Jose Rizal,Region IV-A,2026/07/05,1200.00,Returned
7,1007,Cardo Dalisay,Metro Manila,2026-07-05,NaN,Completed
8,1008,NaN,ncr,2026/07/06,850.75,Completed
9,1005,Ana Roces,calabarzon,2026-07-04,3100.25,Completed


## 1. Dataset Overview — `.info()`

The `info()` method reveals column names, non-null counts, and dtypes. We expect to see missing values in `cust_name` and `amount_paid`, and `object` dtype for `order_date` (indicating it was not parsed as datetime — a sign of mixed formats).

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  12 non-null     int64  
 1   cust_name       10 non-null     str    
 2   region          12 non-null     str    
 3   order_date      12 non-null     str    
 4   amount_paid     10 non-null     float64
 5   status          12 non-null     str    
dtypes: float64(1), int64(1), str(4)
memory usage: 708.0 bytes


### Anomaly Found — Missing Values & Dtype Issues

- `cust_name` has only 10 non-null entries out of 12 rows → **2 missing customer names**
- `amount_paid` has only 10 non-null entries out of 12 rows → **2 missing amounts**
- `order_date` is `object` dtype, not `datetime64` → indicates **mixed date formats** that pandas could not auto-parse
- `transaction_id` is `int64` — no nulls, but duplicates may exist (not visible from info alone)

In [9]:
df.describe(include="all")

,transaction_id,cust_name,region,order_date,amount_paid,status
count,12.000000,10,12,12,10.0000,12
unique,NaN,8,6,10,NaN,3
top,NaN,Juan Dela Cruz,NCR,2026-07-01,NaN,Completed
freq,NaN,2,3,2,NaN,9
mean,1005.083333,NaN,NaN,NaN,2095.2250,NaN
std,3.028901,NaN,NaN,NaN,1348.8760,NaN
min,1001.000000,NaN,NaN,NaN,450.0000,NaN
25%,1002.750000,NaN,NaN,NaN,1275.1250,NaN
50%,1005.000000,NaN,NaN,NaN,1675.2500,NaN
75%,1007.250000,NaN,NaN,NaN,2925.1875,NaN


### Anomaly Found — Statistical Summary

- `describe(include="all")` shows `transaction_id` has 12 entries but only 10 unique values, confirming **duplicate transaction IDs**
- `amount_paid` count is 10 (not 12), confirming **2 missing amounts**; mean is skewed by the high-value 5000.00 order
- `region` has 6 unique values for what should be only 2 actual regions — clear sign of **inconsistent naming/casing**
- `order_date` has 7 unique values but the `top` frequency is 2, which may indicate duplicate rows sharing the same date

In [10]:
null_counts = df.isnull().sum()
null_counts

transaction_id    0
cust_name         2
region            0
order_date        0
amount_paid       2
status            0
dtype: int64

### Anomaly Found — Null Counts

- `cust_name`: **2 nulls** (rows for transaction_id 1003 and 1008 — customers with blank names)
- `amount_paid`: **2 nulls** (rows for transaction_id 1004 and 1007 — missing payment amounts)
- All other columns have 0 nulls
- These nulls will need to be handled in the cleansing pipeline: missing amounts filled with regional median, missing customer names quarantined

## 2. Value Counts — Region & Status

`value_counts()` reveals the distribution of categorical values. For `region`, we expect to see multiple variants of the same logical region (inconsistent casing/naming). For `status`, we expect a small set of order statuses.

In [11]:
df["region"].value_counts()

region
NCR             3
Metro Manila    2
ncr             2
CALABARZON      2
calabarzon      2
Region IV-A     1
Name: count, dtype: int64

### Anomaly Found — Inconsistent Region Naming

The `region` column has **6 distinct raw values** that should map to only **2 actual regions**:

| Raw Value | Should Map To |
|---|---|
| `NCR` | National Capital Region |
| `ncr` | National Capital Region |
| `Metro Manila` | National Capital Region |
| `CALABARZON` | CALABARZON |
| `calabarzon` | CALABARZON |
| `Region IV-A` | CALABARZON |

This inconsistency will cause incorrect GROUP BY aggregations and must be standardized in the cleansing pipeline.

In [12]:
df["status"].value_counts()

status
Completed    9
Cancelled    2
Returned     1
Name: count, dtype: int64